## Input Datasets for this Notebook

Prior to running this notebook, ```depression_depths_calc.ipynb``` and ```netCDF_add_dimension.ipynb``` scripts, in this order, must be ran first. In addition, the following files must be downloaded and pasted into the "Data" folder at the root project directory: 
- [WSC historic discharge data](https://wateroffice.ec.gc.ca/search/historical_e.html): search by outlet station name or number and download Daily data in Date-Data Format (CSV with missing days) (e.g., station number is 05OE006 for Manning)
- [WSC metadata](https://wateroffice.ec.gc.ca/station_metadata/station_characteristics_e.html) of desired outlet point.

Once the Raven input files are generated, the Raven model can be run. The [Raven executable file with NetCDF support](http://raven.uwaterloo.ca/Downloads.html) must be downloaded, and a copy of Raven.exe must be pasted into the folder containing the input files (Data/Raw/manning/Raven_Input_Files). For Windows users, the [NetCDF 4 Library (without DAP)](https://www.unidata.ucar.edu/software/netcdf/docs/winbin.html) must also be installed, and the directory path of the installed NetCDF.dll file must be in your PATH environment variable. For MacOS and linux users, instructions for installing NetCDF can be found in section 2.1 of the [Raven Manual](RavenManual_v3.8.pdf). Once Raven and NetCDF are installed in the correct locations, create a sub-folder within the Raven_Input_Files directory called “input”, and move the GridWeigths.txt file and the runoff netcdf file to this folder. Next, open the command prompt and navigate to your Raven files directory (in Windows, this can be done with “cd c:/Data/Raw/manning/Raven_Input_Files”). Finally, run the following command: “Raven.exe ModelName -o ./output/”, where ModelName is the name of your model, formatted the same as in the names of your Raven input files. The model outputs will be stored in a folder called “output” within the Raven_Input_Files directory. 

## 0. Import required modules

In [1]:
# System
import os
import sys
import fileinput
from pathlib import Path
import shutil


# Analysis
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, LineString, Polygon, MultiPoint


# Time
from datetime import datetime

## 1. Organize data folders and files

### 1.1 Create "Raven" subfolder

In [2]:
# Set watershed name
WATERSHED_NAME = r"Cypress_New"


# Path for this notebook
NOTEBOOK_PATH = Path.cwd()


# Path for PWA root
PWA_PATH = str(NOTEBOOK_PATH.parent.as_posix())


# Path for base raw data folder
BS_DATA_PATH = PWA_PATH + r"/Data/"


# Path for watershed folder (created in depression_depths_calc.ipynb)
WATERSHED_PATH = BS_DATA_PATH + WATERSHED_NAME


# Path for Raven folder
RAVEN_PATH = WATERSHED_PATH + r"/Raven"


# Create Raven folder in specified path
os.makedirs(RAVEN_PATH, 
            exist_ok=True) # Do nothing if already exists

### 1.2 Create subfolders in "Raven" subfolder ("Raw", "Interim", "Processed")

In [3]:
# Specify paths for watershed subfolders
RA_RAW_PATH = RAVEN_PATH + r"/Raw/"
RA_INTERIM_PATH = RAVEN_PATH + r"/Interim/"
RA_PROCESSED_PATH = RAVEN_PATH + r"/Processed/"

In [4]:

# Create watershed subfolders in specified paths
SUBFOLDERS_LIST = [RA_RAW_PATH,
                   RA_INTERIM_PATH,
                   RA_PROCESSED_PATH]
for sub in SUBFOLDERS_LIST:
    os.makedirs(sub, exist_ok=True)

### 1.3 Move input data files to "Raw" subfolder

In [5]:
# Specify source and destination folders before moving files
src = Path(BS_DATA_PATH)
dst = Path(RA_RAW_PATH)


# Move files in base data folder to watershed folder
for file in src.iterdir():
     if file.is_file():
        destination_file = dst / file.name
        if destination_file.exists():
            print(f"Skipped (already exists): {file.name}")
            continue
        shutil.move(str(file), destination_file)

In [6]:
# Depression depths shapefile name (generated in depression_depths_calc.ipynb)
DEPRESSION_DEPTHS_SHAPEFILE = "CLRH_basins_depression_depths"


# Depression depths shapefile path
DEPRESSION_DEPTHS_PATH = WATERSHED_PATH + r"/DepressionDepths/Processed"


# Copy and paste depression depths files to "Raw" folder
for filename in os.listdir(DEPRESSION_DEPTHS_PATH):
    if filename.startswith(DEPRESSION_DEPTHS_SHAPEFILE):
        source_path = os.path.join(DEPRESSION_DEPTHS_PATH, filename)
        destination_path = os.path.join(RA_RAW_PATH, filename)
        if os.path.isfile(source_path):  # Avoid copying directories
            shutil.copy2(source_path, destination_path)  # copy2 preserves metadata

In [7]:
# PHyDAP file name (generated in netCDF_to_PHyDAP.ipynb)
PHYDAP_FILE_NAME = "rdrs_runoff_wlat.nc"


# PHyDAP file path
PHYDAP_FILE_PATH = WATERSHED_PATH + r"/Runoff/Processed"


# Copy and paste PHyDAP file to "Raw" folder
for filename in os.listdir(PHYDAP_FILE_PATH):
    if filename.startswith(PHYDAP_FILE_NAME):
        source_path = os.path.join(PHYDAP_FILE_PATH, filename)
        destination_path = os.path.join(RA_RAW_PATH, filename)
        if os.path.isfile(source_path):  # Avoid copying directories
            shutil.copy2(source_path, destination_path)  # copy2 preserves metadata

## 2. Create Raven input files

### 2.1 Import and project CLRH depression depths file 

In [8]:
# No longer necessary
# Read clrh shapefile
clrh_gdf = gpd.read_file(RA_RAW_PATH + 
                         f"/{DEPRESSION_DEPTHS_SHAPEFILE}.shp")


# # Project clrh gdf to EPSG:3158
# clrh_gdf_project = clrh_gdf.to_crs(4326)

Remove non-contributing lake catchments

In [9]:
clrh_gdf = clrh_gdf[clrh_gdf['Lake_Cat'] < 2]

### 2.2 Calculations

In [10]:
# CALCULATE TIME OF CONCENTRATION AND TIME TO PEAK


# Constants
CN = 63 # Curve number for Manning watershed
CN_RETENTION = (1000 / CN) - 10 # Retention parameter for Manning watershed
SQM_TO_ACRES = 0.00024710538 # Conversion factor from square meters to acres


# Area and slope conversion
clrh_gdf['BasinsArea_acres'] = clrh_gdf['BasArea'] * SQM_TO_ACRES
clrh_gdf['BasSlope_prcnt'] = np.tan(np.radians(clrh_gdf['BasSlope'])) * 100


# Flow length estimation
clrh_gdf['FlowLength_ft'] = 209 * clrh_gdf['BasinsArea_acres'] ** 0.6


# Time of concentration
flow_length = clrh_gdf['FlowLength_ft']
slope = clrh_gdf['BasSlope_prcnt']

clrh_gdf['max_retention'] = CN_RETENTION
clrh_gdf['time_of_conc'] = (flow_length ** 0.8 * (CN_RETENTION + 1) ** 0.7) / \
                                    (1140 * slope ** 0.5)


# Time to peak
clrh_gdf['time_to_peak'] = clrh_gdf['time_of_conc'] * 0.6

In [11]:
# Create lists of necessary columns
subids = list(clrh_gdf['SubId'])
rivlengths = list(clrh_gdf['RivLength'])
hasgauge = list(clrh_gdf['Has_POI'])
dowsubids = list(clrh_gdf['DowSubId'])
basarea = list(clrh_gdf['BasArea'])
elev = list(clrh_gdf['MeanElev'])
lat = list(clrh_gdf['centroid_y'])
lon = list(clrh_gdf['centroid_x'])
slope = list(clrh_gdf['BasSlope'])
aspect = list(clrh_gdf['BasAspect'])
bkfwidth = list(clrh_gdf['BkfWidth'])
bkfdepth = list(clrh_gdf['BkfDepth'])
rivslope = list(clrh_gdf['RivSlope'])
floodpn = list(clrh_gdf['FloodP_n'])
chn = list(clrh_gdf['Ch_n'])
timeofconc = list(clrh_gdf['time_of_conc'])
timetopeak = list(clrh_gdf['time_to_peak'])
depsdepth = list(clrh_gdf['Deps_Depth'])

In [12]:
# MODIFY RIVER LENGTHS AND SLOPES


# Convert river lengths to km
rivlengths_km = [x / 1000 for x in rivlengths]


# Round to 3 decimals
rivlengths_km_rounded = [round(v, 3) for v in rivlengths_km]


# Replace invalid river lengths with 'ZERO-'
rivlengths_mod = list(map(lambda x: 'ZERO-' if x == -0.001 else x,
                          rivlengths_km_rounded))


# Replace invalid river lengths with 0.00
rivlengths_mod_n = list(map(lambda x: 0.0 if x == -0.0012345 else x, \
                            rivlengths_km))


# Modify river slope
rivslope_mod = list(map(lambda x: 0.0001234 if x <= 0.0001 else x, \
                        rivslope))


In [13]:
# Convert basin areas to km2
basarea_km2 = [x / 1000000 for x in \
               basarea]

In [14]:
# Extract outlet subid and drainage area
outlet = clrh_gdf.loc[clrh_gdf['Has_POI'] == 1, 'SubId'].item()
outlet_da = clrh_gdf.loc[clrh_gdf['Has_POI'] == 1, 'DrainArea'].item()


# Convert drainage area to km2
outlet_da_km2 = outlet_da / 1000000

In [15]:
# Replace invalid bankfull depth values with 1
bkfdepth_mod = list(map(lambda x: 1 if x < 1 else x, bkfdepth))

In [16]:
# Replace invalid floodpn and chn values with 0.05 and 0.025, respectively
floodpn_mod = list(map(lambda x: 0.05 if x < 0 else x, \
                floodpn))
chn_mod = list(map(lambda x: 0.025 if x < 0 else x, \
               chn))

Specify start and end datetime

In [17]:
# Specify model start time
model_start_str = input("Enter model start time (YYYY-MM-DD HH:MM:SS): ")  
if not model_start_str:
    model_start_str = "1980-01-04 00:00:00"
if len(model_start_str) != 19 or model_start_str[4] != '-' or model_start_str[7] != '-' or \
        model_start_str[10] != ' ' or model_start_str[13] != ':' or model_start_str[16] != ':':
    raise ValueError("Start date must be in the format YYYY-MM-DD HH:MM:SS")

# Specify model end time
model_end_str = input("Enter model end time (YYYY-MM-DD HH:MM:SS): ")
if not model_end_str:
    model_end_str = "2017-12-30 00:00:00"
if len(model_end_str) != 19 or model_end_str[4] != '-' or model_end_str[7] != '-' or \
        model_end_str[10] != ' ' or model_end_str[13] != ':' or model_end_str[16] != ':':
    raise ValueError("End date must be in the format YYYY-MM-DD HH:MM:SS")

### 2.3 Create RVT File

In [18]:
# WSC metadata file name
WSC_METADATA_FILE_NAME = "metadata_20250911T1922"


# Read WSC metadata file
wsc_metadata_gdf = gpd.read_file(RA_RAW_PATH + WSC_METADATA_FILE_NAME + ".csv")


# WSC data file name
WSC_DATA_FILE_NAME = "daily_20250911T1922"


# Read WSC data file
wsc_data_raw = pd.read_csv(RA_RAW_PATH + WSC_DATA_FILE_NAME + ".csv",
                          skiprows=1)


# # Extract discharge values from dataframe and replace NaN values with -1.2345
# wsc_data_dschrg = wsc_data_df.loc[wsc_data_df['PARAM']==1, 'Value']
# wsc_data_dschrg.fillna(-1.2345, inplace=True)

In [19]:

## Process WSC timeseries
# Remove rows outside model start and end time 
wsc_time_check = (pd.to_datetime(wsc_data_raw['Date']) >= pd.to_datetime(datetime.strptime(model_start_str, '%Y-%m-%d %H:%M:%S'))) & \
                   (pd.to_datetime(wsc_data_raw['Date']) <= pd.to_datetime(datetime.strptime(model_end_str, '%Y-%m-%d %H:%M:%S')))

wsc_data_clip = wsc_data_raw[wsc_time_check]



In [20]:
# Select only discharge data (PARAM == 1) 
discharge_df = wsc_data_clip[wsc_data_clip['PARAM'] == 1].copy()

# Collate a list of data gaps
data_gaps = discharge_df[discharge_df['Value'].isnull()]['Date']

data_gaps = pd.to_datetime(data_gaps, format='%Y/%m/%d')


data_gap_forward = (data_gaps.shift(-1) - data_gaps) > pd.Timedelta(days=1.5)
data_gap_backward = (data_gaps - data_gaps.shift(1)) > pd.Timedelta(days=1.5)

data_gap_limits = data_gaps[data_gap_forward | data_gap_backward]

single_gaps = data_gaps[data_gap_forward & data_gap_backward]

data_gap_limits = pd.concat([data_gap_limits, single_gaps]).sort_values().reset_index(drop=True)

wsc_null_start = np.isnan(discharge_df['Value'].iloc[0])
wsc_null_end = np.isnan(discharge_df['Value'].iloc[-1])

if wsc_null_start:
    data_gap_limits = pd.concat([pd.Series(pd.to_datetime(model_start_str, format = "%Y-%m-%d %H:%M:%S")), data_gap_limits])

if wsc_null_end:
    data_gap_limits = pd.concat([data_gap_limits, pd.Series(pd.to_datetime(model_end_str, format = "%Y-%m-%d %H:%M:%S"))])

data_gap_limits = data_gap_limits.reset_index(drop=True)

In [21]:
## Generate the time gaps string for the rvt file
wsc_gaps_string = ""
for i in range(0, len(data_gap_limits), 2):
    if wsc_null_start:
        start_gap = data_gap_limits.iloc[i]
        end_gap = data_gap_limits.iloc[i+1]
        wsc_gaps_string += f"# Missing data period {i//2 + 1} = {start_gap.strftime('%Y-%m-%d')} - {end_gap.strftime('%Y-%m-%d')} \n"

print(wsc_gaps_string)

# Missing data period 1 = 1980-01-04 - 1980-02-29 
# Missing data period 2 = 1980-11-01 - 1981-02-28 
# Missing data period 3 = 1981-11-01 - 1982-02-28 
# Missing data period 4 = 1982-11-01 - 1983-02-28 
# Missing data period 5 = 1983-11-01 - 1984-02-29 
# Missing data period 6 = 1984-11-01 - 1985-02-28 
# Missing data period 7 = 1985-11-01 - 1986-02-28 
# Missing data period 8 = 1986-11-01 - 1987-02-28 
# Missing data period 9 = 1987-11-01 - 1988-02-29 
# Missing data period 10 = 1988-11-01 - 1989-02-28 
# Missing data period 11 = 1989-11-01 - 1990-02-28 
# Missing data period 12 = 1990-11-01 - 1991-02-28 
# Missing data period 13 = 1991-11-01 - 1992-02-25 
# Missing data period 14 = 1992-11-01 - 1993-02-28 
# Missing data period 15 = 1993-11-01 - 1994-02-28 
# Missing data period 16 = 1994-11-01 - 1995-02-28 
# Missing data period 17 = 1995-11-01 - 1996-02-29 
# Missing data period 18 = 1996-11-01 - 1997-02-28 
# Missing data period 19 = 1997-11-01 - 1998-02-28 
# Missing data period

In [22]:
## Deprecated - hourly interpolation in rvt causes issues with Raven

# # Convert 'Date' to datetime if not already
# discharge_df['Date'] = pd.to_datetime(discharge_df['Date'])

# # Shift each date forward by 12 hours (to noon)
# discharge_df['Date'] = discharge_df['Date'] + pd.Timedelta(hours=12)

# # Set Date as index for interpolation
# discharge_df.set_index('Date', inplace=True)

# # Resample to hourly frequency and interpolate
# hourly_discharge = discharge_df['Value'].resample('H').asfreq()
# hourly_discharge = hourly_discharge.interpolate('time', limit_area = 'inside', limit = 24)

# # Reset index if needed
# hourly_discharge = hourly_discharge.reset_index()

# hourly_discharge.Value = hourly_discharge.Value.round(3)

# hourly_discharge.fillna(-1.2345, inplace=True)

discharge_df.fillna(-1.2345, inplace=True)

In [23]:
start_str = model_start_str.split(" ")[0]
end_str = model_end_str.split(" ")[0]

# Create file and store in processed folder
FILE_PATH_RVT = RA_PROCESSED_PATH + "WSC" + \
            wsc_metadata_gdf['Station Number'].values[0] + ".rvt"


# Extract relevant info from WSC metadata file
station_name = wsc_metadata_gdf['Station Name'].iloc[0]
station_id = wsc_metadata_gdf['Station Number'].iloc[0]
num_points = len(discharge_df)


# Write metadata to file
with open(FILE_PATH_RVT, 'w') as file:
    header = f"""\
#################################################################################
# Name = {station_name}
# ID = {station_id}
# Drainage Area = {int(outlet_da_km2)} km2
# Sub-basin ID = {outlet}
# Natural/Reference = Regulated
# Country = CA
# Date Range = {start_str} - {end_str}
# Data time-step = 24 hr
{wsc_gaps_string}
# Missing data are filled with -1.2345
##################################################################################
:ObservationData	HYDROGRAPH	{outlet}	m3/s
    {model_start_str}	1	{num_points}
"""
    file.write(header)


# Write data to file
with open(FILE_PATH_RVT, 'a') as file:
    for value in discharge_df['Value']:
        file.write(f"\t{value:.4f}\n")
    file.write(':EndObservationData')


### 2.4 Create RVI File

In [ ]:
# Create file and store in processed folder
FILE_PATH_RVI = RA_PROCESSED_PATH + WATERSHED_NAME.replace(' ', '') + ".rvi"


# Declare model run time
model_run_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')


# Write to file
with open(FILE_PATH_RVI, 'w') as file:
    file.write(f"""\
# ----------------------------------------------
# Raven Input file
# {WATERSHED_NAME}
# ----------------------------------------------
:StartDate             1980-01-03 00:00:00           # Model run start time, format in yyyy-mm-dd hh:mm:ss
:EndDate               2020-12-30 00:00:00           # Model run end time, format in yyyy-mm-dd hh:mm:ss
:Method                ORDERED_SERIES                # Numerical method used for simulation
:TimeStep              01:00:00                      # Expressed in days as a real-valued number (e.g., 1.0 for one day) or using hh:mm:ss format (e.g., 24:00:00 for one hour).
:RunName               {WATERSHED_NAME:<30}# The name of model run. It will be added as a prefix of model output files.

:CatchmentRoute        ROUTE_TRI_CONVOLUTION         # Catchment routing method, used to convey water from the catchment tributaries and rivulets to the subbasin outlets. DEFAULT ROUTE_DUMP, which instantly ‘dumps’ all water in the subbasin stream reach.
:Routing               ROUTE_DIFFUSIVE_WAVE          # Channel routing method which is used to transport water from upstream to downstream within the main subbasin channels. DEFAULT ROUTE_DIFFUSIVE_WAVE, which analytically solves the diffusive wave equation along the reach using a constant reference celerity.
:PrecipIceptFract      PRECIP_ICEPT_NONE             # Estimation of the precipitation interception fraction. In this routing model, stream input(s) are "pretending" to be precipitation going into Raven, thus using DEFAULT PRECIP_ICEPT_NONE to indicate no interception processes are adopted.
:PotentialMeltMethod   POTMELT_NONE                  # Estimation of the potential snow melt. In this routing model, snow melt processes are not relevant, thus using DEFAULT POTMELT_NONE method.
:SWRadiationMethod     SW_RAD_NONE                   # Estimation of shortwave incoming radiation. This is turned off for a routing-only model.
:SoilModel             SOIL_ONE_LAYER                # In this routing model, use DEFAULT SOIL_ONE_LAYER to define single soil layer structure.

:HydrologicProcesses
  :Precipitation     PRECIP_RAVEN            ATMOS_PRECIP     MULTIPLE          # Moves stream input(s) from ATMOS_PRECIP to PONDED_WATER storage (waiting for runoff). Use DEFAULT PRECIP_RAVEN method.
  :Abstraction       ABST_FILL               PONDED_WATER     DEPRESSION
  :Flush             RAVEN_DEFAULT           PONDED_WATER     SURFACE_WATER         # Moves water from PONDED_WATER to SURFACE_WATER (routed to outlet). Use DEFAULT RAVEN_DEFAULT method.
  :Seepage           SEEP_LINEAR             DEPRESSION       SOIL[0]
:EndHydrologicProcesses

# Defines the hydrograph performance metrics output by Raven. Either one or multiple is acceptable.
:EvaluationMetrics           NASH_SUTCLIFFE    PCT_BIAS    KLING_GUPTA
""")

### 2.5 Create RVC File

In [25]:
# Create file and store in processed folder
FILE_PATH_RVC = RA_PROCESSED_PATH + WATERSHED_NAME.replace(' ', '') + ".rvc"


# Write to file
with open(FILE_PATH_RVC, 'w') as file:
    # Write initial conditions to file
    file.write(f"""\
# ----------------------------------------------
# Raven Input file
# {WATERSHED_NAME}
# ----------------------------------------------

:UniformInitialConditions SOIL[0] 15            # Initial conditions for the groundwater storage, in mm. DEFAULT 15 mm.
""")


### 2.6 Create RVP File

In [26]:
# Create file and store in processed folder
FILE_PATH_RVP = RA_PROCESSED_PATH + WATERSHED_NAME.replace(' ', '') + ".rvp"


# Write to file
with open(FILE_PATH_RVP, 'w') as file:
    # Write soil, vegetation, and land use content
    file.write(f"""\
# ----------------------------------------------
# Raven Input file
# {WATERSHED_NAME}
# ----------------------------------------------

# Soil/Vegetation/Land use class definitions ------
:SoilClasses
  AQUIFER      # only one soil class used here
:EndSoilClasses

:SoilProfiles
  # name, number of layers, soil class, thickness [m]
  LandHRU_soil_profile,    1,    AQUIFER, 5
:EndSoilProfiles

# The vegetation and land use parameters unused in routing
:VegetationClasses
  :Attributes,                MAX_HT,       MAX_LAI,    MAX_LEAF_COND
  :Units,                       m,            none,       mm_per_s
  LandHRU_vege_class,          25,           5.0,          5.0
:EndVegetationClasses

:LandUseClasses
  :Attributes,        IMPERMEABLE_FRAC,         FOREST_COVERAGE
  :Units,                     fract,                    fract
  LandHRU_landuse_class,      0,                        1""")
    


# Append to file
with open(FILE_PATH_RVP, 'a') as file:
    # Append land use parameters for each subbasin
    for subid in subids:
        file.write(f"\n  SUB{subid},              0,                        1")

    file.write("""\
\n:EndLandUseClasses

# Global Parameters-----------------------------
:AvgAnnualRunoff        75

:LandUseParameterList
  :Parameters     ,          DEP_SEEP_K,             DEP_MAX
  :Units          ,               1/d,                   mm
    [DEFAULT]     ,             0.01,              20.0000""")

    # Write subbasin-specific parameter values
    for subid, depth in zip(subids, 
                            depsdepth):
        file.write(f"\n SUB{subid:<12},             0.01,          {depth:.4f}")

    file.write("""\

:EndLandUseClasses

# redirects to channel properties file
:RedirectToFile         channel_properties.rvp
""")

### 2.7 Create RVT File

In [ ]:
# Create file and store in processed folder
FILE_PATH_RVT = RA_PROCESSED_PATH + WATERSHED_NAME.replace(' ', '') + ".rvt"


# Write to file
with open(FILE_PATH_RVT, 'w') as file:
    # Write PHyDAP netCDF metadata to file and point to GridWeights and WSC files
    file.write(f"""\
# ----------------------------------------------
# Raven Input file
# {WATERSHED_NAME}
# ----------------------------------------------

:GriddedForcing           StreamInputs
    :ForcingType          PRECIP
    :FileNameNC           input/{PHYDAP_FILE_NAME}
    :VarNameNC            runoff
    :DimNamesNC           lon lat time     # must be in the order of (x,y,t)
    :LinearTransform      24.0     0.0
    :RedirectToFile       input/GridWeights.txt
:EndGriddedForcing

# NOTE: the Linear Transform converts mm/1hr --> mm/d

:GriddedForcing           Averagetemp
    :ForcingType          TEMP_AVE
    :FileNameNC           input/{PHYDAP_FILE_NAME}
    :VarNameNC            runoff
    :DimNamesNC           lon lat time
    :RedirectToFile       input/GridWeights.txt
:EndGriddedForcing

# Observed Discharge Data-----------------------
:RedirectToFile           WSC{wsc_metadata_gdf['Station Number'].values[0]}.rvt
""")

### 2.8 Create RVH File

In [ ]:
# Create file and store in processed folder
FILE_PATH_RVH = RA_PROCESSED_PATH + WATERSHED_NAME.replace(' ', '') + ".rvh"


# Write to file
with open(FILE_PATH_RVH, 'w') as file:
    # Write SubBasin metadata to file
    file.write(f"""\
# ----------------------------------------------
# Raven Input file
# {WATERSHED_NAME}
# ----------------------------------------------

## SubBasin List---------------------------------------------------------------------
# ID:            A positive long integer identifier unique to this subbasin.
# NAME:          A nickname for subbasin. DEFAULT string("sub_" + ID).
# DOWNSTREAM_ID: The ID of the subbasin that receives this subbasin's outflowing waters.
# PROFILE:       The representative channel profile code (defined in channel_properties.rvp).
# REACH_LENGTH:  The length of the primary reach channel in the basin (in km).
# GAUGED:        Flag to indicate whether hydrographs for this subbasin are output (1 = yes, 0 = no).
#                In this routing model, both subbasins with reservoirs and flow gauges are GAUGED = 1.

:SubBasins
  :Attributes     ID     NAME     DOWNSTREAM_ID     PROFILE     REACH_LENGTH     GAUGED
  :Units          none   none     none              none        km              none
""")
    

# Append to file
with open(FILE_PATH_RVH, 'a') as file:
    # Append subbasin data to file
    for i in range(len(subids)):
        subid = subids[i]
        downstream_id = dowsubids[i]
        channel_profile = f"chn_{subid}"
        reach_length = list(rivlengths_mod)[i]
        gauged = hasgauge[i]

        reach_length_fmt = f"{reach_length:>10}"  if isinstance(reach_length, str) else f"{reach_length:>10.2f}"
        file.write(
            f"{subid:>10}  "                       # Subbasin ID
            f"{'sub' + str(subid):<15}  "          # Subbasin name
            f"{downstream_id:>10}  "               # Downstream Subbasin ID
            f"{channel_profile:<15}  "             # Channel profile name
            f"{reach_length_fmt}  "                # Reach length (km)
            f"{gauged:>5}\n"                       # Gauged flag (1 or 0)
        )


# Append to file
with open(FILE_PATH_RVH, 'a') as file:
    # Append HRU metadata to file
    file.write(
        ":EndSubBasins\n\n"
        "## HRU List ---------------------------------------------------------------------\n"
        "# HRU ID:          A positive integer unique to this HRU. Note that more than one HRUs\n"
        "#                 may have the same subbasin ID, but HRU ID is unique.\n"
        "# AREA:            HRU area value, in km².\n"
        "# ELEVATION:       HRU mean elevation, in m.\n"
        "# LATITUDE:        Latitude of HRU centroid, in decimal degrees.\n"
        "# LONGITUDE:       Longitude of HRU centroid, in decimal degrees.\n"
        "# BASIN_ID:        ID of the subbasin in which the HRU is located.\n"
        "# LAND_USE_CLASS:  Land use class defined in .rvp file. DEFAULT LandHRU_landuse_class.\n"
        "# VEG_CLASS:       Vegetation class defined in .rvp file. DEFAULT LandHRU_vege_class.\n"
        "# SOIL_PROFILE:    Soil profile defined in .rvp file. DEFAULT LandHRU_soil_profile.\n"
        "# AQUIFER_PROFILE: Aquifer class. DEFAULT [NONE] in this routing model.\n"
        "# TERRAIN_CLASS:   Terrain class. DEFAULT [NONE] in this routing model.\n"
        "# SLOPE:           HRU mean slope, in degrees.\n"
        "# ASPECT:          HRU mean aspect, in degrees. North: 0°, West: 90°, South: 180°, East: 270°.\n"
        "#\n"
        ":HRUs\n"
        "  :Attributes     AREA     ELEVATION     LATITUDE     LONGITUDE     BASIN_ID     LAND_USE_CLASS         "
        "VEG_CLASS          SOIL_PROFILE     AQUIFER_PROFILE     TERRAIN_CLASS     SLOPE     ASPECT\n"
        "  :Units          km2      m             deg          deg           none         none                   "
        "none                none             none               none              deg       degN\n"
    )


# Append to file
with open(FILE_PATH_RVH, 'a') as file: 
    # Append HRU data to file
    for i in range(len(subids)):
        subid = subids[i]
        area = round(basarea_km2[i], 4)
        elevation = round(elev[i], 2)
        latitude = round(lat[i], 6)
        longitude = round(lon[i], 6)
        slope_val = round(slope[i], 2)
        aspect_val = round(aspect[i], 2)

        file.write(
            f"{subid:>15}"
            f"{area:>10}"
            f"{elevation:>10}"
            f"{latitude:>15}"
            f"{longitude:>15}"
            f"{subid:>15}"
            f"{'LandHRU_landuse_class':>30}"
            f"{'LandHRU_vege_class':>25}"
            f"{'LandHRU_soil_profile':>30}"
            f"{'[NONE]':>10}"
            f"{'[NONE]':>10}"
            f"{slope_val:>10}"
            f"{aspect_val:>10}\n"
        )

    file.write(':EndHRUs\n')


# Append to file
# Open file in append mode
with open(FILE_PATH_RVH, 'a') as file:
    # Append time of concentration and time to peak data to file
    file.write(
        '\n# SubBasin Properties ----------------------------------------------------\n'
        ':SubBasinProperties\n'
        '  :Parameters      TIME_CONC  TIME_TO_PEAK\n'
        '  :Units               hrs           hrs\n'
    )

    for i in range(len(subids)):
        subid = subids[i]
        time_conc = round(timeofconc[i], 2)
        time_peak = round(timetopeak[i], 2)

        file.write(f"{subid:>15}{time_conc:>10}{time_peak:>14}\n")

    file.write(':EndSubBasinProperties\n')

### 2.9 Create channel properties RVP file

In [ ]:
# Create file and store in processed folder
FILE_PATH_RVP_CHANNEL = RA_PROCESSED_PATH + "channel_properties.rvp"


# Write to file
with open(FILE_PATH_RVP_CHANNEL, 'w') as file:
    # Write header content to file
    header = (
        "# ----------------------------------------------\n"
        f"# Raven Input file\n"
        f"# {WATERSHED_NAME}\n"
        "# ----------------------------------------------\n"
    )
    file.write(header)


# Append to file
with open(FILE_PATH_RVP_CHANNEL, 'a') as file:
    # Append channel transect data for each subbasin to file
    for i in range(len(subids)):
        # Define survey point x values based on bankfull depth and width
        x1 = 0
        x2 = bkfdepth_mod[i] * 2
        x3 = bkfwidth[i] * 2
        x4 = x2 + x3 * 1.5
        x5 = x2 + x3 * 2
        x6 = x2 * 2 + x3 * 2
        x7 = x3 * 3 + x2 
        x8 = x3 * 3 + x2 * 2

        # Prepare formatted multiline string for this subbasin
        content = (
            f"\n###################################chn_{subids[i]}"
            f"\n## Subbasin ID:        {subids[i]}"
            f"\n## Subbasin area:      {round(basarea_km2[i], 2)} km2"
            f"\n## Channel length:     {round(rivlengths_mod_n[i], 2)} km"
            f"\n## Channel elevation:  {round(elev[i], 3)} m"
            f"\n## Bankfull width:     {round(bkfwidth[i], 4)} m"
            f"\n## Bankfull depth:     {round(bkfdepth_mod[i], 2)} m"
            f"\n:ChannelProfile       chn_{subids[i]}"
            f"\n    :Bedslope         {round(rivslope_mod[i], 7)}"
            f"\n    :SurveyPoints"
            f"\n        {x1: >5}  {round(elev[i] + bkfdepth_mod[i] + (bkfdepth_mod[i] / 2), 2)}"
            f"\n        {round(x2, 2): >5}  {round(elev[i] + bkfdepth_mod[i], 2)}"
            f"\n        {round(x3, 2): >5}  {round(elev[i] + bkfdepth_mod[i], 2)}"
            f"\n        {round(x4, 2): >5}  {round(elev[i], 2)}"
            f"\n        {round(x5, 2): >5}  {round(elev[i], 2)}"
            f"\n        {round(x6, 2): >5}  {round(elev[i] + bkfdepth_mod[i], 2)}"
            f"\n        {round(x7, 2): >5}  {round(elev[i] + bkfdepth_mod[i], 2)}"
            f"\n        {round(x8, 2): >5}  {round(elev[i] + bkfdepth_mod[i] + (bkfdepth_mod[0] / 2), 2)}"
            f"\n    :EndSurveyPoints"
            f"\n    :RoughnessZones"
            f"\n        {0: >5} {round(floodpn_mod[i], 6)}"
            f"\n        {round(x3, 2): >5} {round(chn_mod[i], 6)}"
            f"\n        {round(x6, 2): >5} {round(floodpn_mod[i], 6)}"
            f"\n    :EndRoughnessZones"
            f"\n:EndChannelProfile\n"
        )

        # Write the content to the file
        file.write(content)


### 2.10 Create GridWeights TXT file

In [ ]:
# # Check the cell # corresponding to the relevant HYBAS_IDs
# from netCDF4 import Dataset
# import numpy as np

# nc_file = Dataset("../Data/Cypress_New/Runoff/Raw/PHyDAP_RDRS21_hourly_runoff.nc", "r") # Open the netCDF file in read mode

# # hybas_ids = nc_file.variables['HYBAS_ID'][:] # Extract the HYBAS_ID variable

# # def find_cell(hybas_ids, target_id):
# #     for idx, val in np.ndenumerate(hybas_ids):
# #         if val == target_id:
# #             return idx
# #     return None

# # for target_id in [7120966470,7120966730, 7120298981, 7120299860, 7120965000]:
# #     print(f"Cell number for HYBAS_ID {target_id}: {find_cell(hybas_ids, target_id)}")

# nc_file.close()  # Close the netCDF file

In [ ]:
# Create file and store in processed folder
FILE_PATH_GRIDWEIGHTS = RA_PROCESSED_PATH + "GridWeights.txt"


# Write to file
with open(FILE_PATH_GRIDWEIGHTS, 'w') as file:
    # Write header and GridWeights metadata to file
    file.write(
        f"# ----------------------------------------------\n"
        f"# Raven Input file\n"
        f"# {WATERSHED_NAME}\n"
        f"# ----------------------------------------------\n\n"
        f":GridWeights\n"
        f"   #\n"
        f"   # [# HRUs]\n"
        f"   :NumberHRUs       {len(subids)}\n"
        f"   :NumberGridCells  4175\n"
        f"   #\n"
        f"   # [HRU ID] [Cell #] [w_kl]"
    )


# Open file in append mode
with open(FILE_PATH_GRIDWEIGHTS, 'a') as file:
    # Append grid weights data to file
    for subid in subids:
        file.write(f"\n  {subid:>10}{108:>10}{1.0:>10}")
    file.write("\n:EndGridWeights")

### 3. Create template files for RAVEN calibration workflow (Optional)

#### 3.1 Create .rvh template file

In [ ]:
import re

def generate_rvh_template(real_rvh_path, tpl_rvh_path):
    with open(real_rvh_path, "r") as f:
        lines = f.readlines()

    template_lines = []
    in_subbasins = False
    in_hrus = False
    in_props = False

    for line in lines:
        # Track sections
        if line.strip().startswith(":SubBasins"):
            in_subbasins = True
            in_hrus = in_props = False
        elif line.strip().startswith(":HRUs"):
            in_hrus = True
            in_subbasins = in_props = False
        elif line.strip().startswith(":SubBasinProperties"):
            in_props = True
            in_subbasins = in_hrus = False
        elif line.strip().startswith(":EndSubBasins") or \
             line.strip().startswith(":EndHRUs") or \
             line.strip().startswith(":EndSubBasinProperties"):
            in_subbasins = in_hrus = in_props = False

        # --- SubBasins section ---
        if in_subbasins:
            # Update the header lines to include Q_REFERENCE
            if line.strip().startswith(":Attributes"):
                line = line.rstrip("\n") + "   Q_REFERENCE\n"
            elif line.strip().startswith(":Units"):
                line = line.rstrip("\n") + "    m3/s\n"

            # Update data lines
            elif not line.strip().startswith(":") and line.strip() and not line.strip().startswith("#"):
                parts = line.split()
                subid, name, downstream, profile, reach_length, gauged = parts[:6]
                qref = f"qref_{subid}"
                line = (
                    f"{int(subid):>10}  sub{subid:<10}  {int(downstream):>10}  "
                    f"chn_{subid:<10}  {reach_length:>10}  {gauged:>5}   {qref}\n"
                )

        # --- HRUs section ---
        if in_hrus and not line.strip().startswith(":") and line.strip() and not line.strip().startswith("#"):
            parts = line.split()
            if len(parts) >= 6:
                hruid, area, elev, lat, \
                lon, basin_id, land_use_class, \
                veg_class, soil_profile, \
                aquifer_profile, \
                terrain_class, slope, aspect = parts
                # line = (
                #     f"{int(hruid):>10}  H{hruid}_area     {elev:>10}  {lat:>12}  {lon:>12}  "
                #     f"{int(basin_id):>8}   SUB{basin_id:<10}   {land_use_class}   {veg_class}   "
                #     f"{soil_profile}   {aquifer_profile}   {terrain_class}   {slope}   {float(aspect):>10.2f}\n"
                # )
                line = (
                        f"{int(hruid):12d}      "                   # HRU ID
                        f"H{hruid}_area     "                       # AREA
                        f"{float(elev):10.2f}      "                # ELEVATION
                        f"{float(lat):14.6f}     "                  # LATITUDE
                        f"{float(lon):14.6f}      "                       # LONGITUDE
                        f"{int(basin_id):>8}           "            # BASIND_ID 
                        f"SUB{basin_id:<10}      "                  # LAND_USE_CLASS
                        f"{veg_class:>20}           "               # VEG_CLASS
                        f"{soil_profile:>25}   "                    # SOIL_PROFILE
                        f"{aquifer_profile:>10}   "                 # AQUIFER_PROFILE
                        f"{terrain_class:>10}       "               # TERRAIN_CLASS
                        f"{float(slope):10.2f}     "                # SLOPE
                        f"{float(aspect):10.2f}\n"                  # ASPECT
                    )

        # --- SubBasinProperties section ---
        if in_props and not line.strip().startswith(":") and line.strip() and not line.strip().startswith("#"):
            parts = line.split()
            if len(parts) >= 1:
                subid = parts[0]
                line = f"{int(subid):>10}     tc_{subid:<10}    tp_{subid}\n"

        template_lines.append(line)

    # Save template
    with open(tpl_rvh_path, "w") as f:
        f.writelines(template_lines)

    print(f"Template file saved at {tpl_rvh_path}")



In [ ]:
FILE_PATH_RVH_TPL = RA_PROCESSED_PATH + f"{WATERSHED_NAME}.rvh.tpl"
FILE_PATH_RVH = RA_PROCESSED_PATH + WATERSHED_NAME.replace(' ', '') + ".rvh"
generate_rvh_template(real_rvh_path=FILE_PATH_RVH, tpl_rvh_path=FILE_PATH_RVH_TPL)

#### 3.2 Create .rvp template file

In [ ]:
def generate_rvp_template(real_rvp_path: str, tpl_rvp_path: str, subids: list[int], 
                          channel_props_filename: str):
    """Generate an .rvp template file based on the given subids."""

    with open(tpl_rvp_path, "w") as f:
        f.write("# ----------------------------------------------\n")
        f.write("# Raven Input file\n")
        f.write(f"# {WATERSHED_NAME}\n")
        f.write("# ----------------------------------------------\n\n")

        # Soil classes
        f.write("# Soil/Vegetation/Land use class definitions ------\n")
        f.write(":SoilClasses\n")
        f.write("  AQUIFER      # only one soil class used here\n")
        f.write(":EndSoilClasses\n\n")

        f.write(":SoilProfiles\n")
        f.write("  # name, number of layers, soil class, thickness [m]\n")
        f.write("  LandHRU_soil_profile,    1,    AQUIFER, 5\n")
        f.write(":EndSoilProfiles\n\n")

        # Vegetation
        f.write("# the vegetation and land use parameters unused in routing\n")
        f.write(":VegetationClasses\n")
        f.write("  :Attributes,                MAX_HT,       MAX_LAI,    MAX_LEAF_COND\n")
        f.write("  :Units,                       m,            none,       mm_per_s\n")
        f.write("  LandHRU_vege_class,           25,           5.0,          5.0\n")
        f.write(":EndVegetationClasses\n\n")

        # LandUseClasses
        f.write(":LandUseClasses\n")
        f.write("  :Attributes,        IMPERMEABLE_FRAC,         FOREST_COVERAGE\n")
        f.write("  :Units,                     fract,                    fract\n")
        f.write("  LandHRU_landuse_class,      0,                        1\n")
        for subid in subids:
            f.write(f"  SUB{subid},\t      0,                        1\n")
        f.write(":EndLandUseClasses\n\n")

        # Global parameter
        f.write("# Global Parameters-----------------------------\n")
        f.write(":AvgAnnualRunoff \t\t75\n\n")

        # LandUseParameterList
        f.write(":LandUseParameterList\n")
        f.write("  :Parameters,\tDEP_SEEP_K,\t\tDEP_MAX\n")
        f.write("  :Units     ,\t       1/d,\t\t     mm\n")
        f.write("    [DEFAULT],          seep_fact,        20.000\n")
        for subid in subids:
            f.write(f"    SUB{subid}   ,\tseep_fact,\t      s{subid}_dep\n")
        f.write(":EndLandUseParameterList\n\n")

        # Redirect
        f.write("# redirects to channel properties file\n")
        f.write(f":RedirectToFile         {channel_props_filename}.rvp\n")



In [ ]:
def extract_subids_from_rvh(rvh_path):
    subids = []
    with open(rvh_path, "r") as f:
        for line in f:
            if line.strip() and not line.startswith(":") and not line.startswith("#"):
                parts = line.split()
                # SubID is always the first column after :Attributes and :Units
                if parts[0].isdigit():
                    subids.append(int(parts[0]))
    return list(set(subids))

subids = extract_subids_from_rvh(FILE_PATH_RVH)


In [33]:
FILE_PATH_RVP_TPL = RA_PROCESSED_PATH + f"{WATERSHED_NAME}.rvp.tpl"
generate_rvp_template(real_rvp_path=FILE_PATH_RVP, tpl_rvp_path=FILE_PATH_RVP_TPL, 
                      subids=subids, channel_props_filename="channel_properties")

#### 3.3 Create channel properties .rvp template file

In [ ]:
def generate_channel_rvp_template(real_rvp_path: str, tpl_rvp_path: str):
    """
    Generate a channel properties .rvp template file by copying the original.
    
    Parameters:
    -----------
    real_rvp_path : str
        Path to the existing channel properties .rvp file.
    tpl_rvp_path : str
        Path where the .tpl version should be saved.
    """
    # Ensure the output directory exists
    os.makedirs(os.path.dirname(tpl_rvp_path), exist_ok=True)

    # Copy the file content
    shutil.copy(real_rvp_path, tpl_rvp_path)
    print(f"Template created: {tpl_rvp_path}")

In [ ]:
FILE_PATH_RVP_CHANNEL_TPL = RA_PROCESSED_PATH + f"channel_properties.rvp.tpl"
generate_channel_rvp_template(real_rvp_path=FILE_PATH_RVP_CHANNEL, tpl_rvp_path=FILE_PATH_RVP_CHANNEL_TPL)

#### 3.4 Create CSVs containing other specific info from .rvh file

In [ ]:
def extract_hru_areas(rvh_path: str, output_csv: str):
    """
    Extract HRU IDs and areas from a .rvh file and save to a CSV.
    
    Parameters:
    -----------
    rvh_path : str
        Path to the .rvh file.
    output_csv : str
        Path to the output CSV file.
    """
    hru_data = []
    in_hru_section = False

    with open(rvh_path, 'r') as file:
        for line in file:
            line = line.strip()
            
            # Check if we are in the HRU section
            if line.startswith(":HRUs"):
                in_hru_section = True
                continue
            elif line.startswith(":EndHRUs"):
                in_hru_section = False
                continue
            
            if in_hru_section:
                # Skip header lines
                if line.startswith(":Attributes") or line.startswith(":Units") or line == "":
                    continue
                
                # Split line by whitespace
                parts = line.split()
                if len(parts) >= 2:
                    hru_id = parts[0]
                    area = parts[1]
                    hru_data.append([hru_id, area])
    
    # Convert to DataFrame and save as CSV
    df = pd.DataFrame(hru_data, columns=["HRU", "AREA"])
    df.to_csv(output_csv, index=False)
    print(f"HRU areas saved to: {output_csv}")

In [ ]:
FILE_PATH_HRU_AREAS_CSV = RA_PROCESSED_PATH + f"{WATERSHED_NAME}_HRU_areas.csv"
extract_hru_areas(rvh_path=FILE_PATH_RVH, output_csv=FILE_PATH_HRU_AREAS_CSV)

In [ ]:
def extract_subbasin_properties(rvh_path: str, output_csv: str):
    """
    Extract SubBasin IDs and their TIME_CONC (tc) and TIME_TO_PEAK (tp) from a .rvh file 
    and save to a CSV.

    Parameters:
    -----------
    rvh_path : str
        Path to the .rvh file.
    output_csv : str
        Path to the output CSV file.
    """
    subbasin_data = []
    in_subbasin_section = False

    with open(rvh_path, 'r') as file:
        for line in file:
            line = line.strip()
            
            # Check if we are in the SubBasinProperties section
            if line.startswith(":SubBasinProperties"):
                in_subbasin_section = True
                continue
            elif line.startswith(":EndSubBasinProperties"):
                in_subbasin_section = False
                continue
            
            if in_subbasin_section:
                # Skip header lines
                if line.startswith(":Parameters") or line.startswith(":Units") or line == "":
                    continue
                
                # Split line by whitespace
                parts = line.split()
                if len(parts) >= 3:
                    subid = parts[0]
                    tc = parts[1]
                    tp = parts[2]
                    subbasin_data.append([subid, tc, tp])
    
    # Convert to DataFrame and save as CSV
    df = pd.DataFrame(subbasin_data, columns=["SubId", "tc", "tp"])
    df.to_csv(output_csv, index=False)
    print(f"SubBasin properties saved to: {output_csv}")


In [ ]:
FILE_PATH_TP_TC_CSV = RA_PROCESSED_PATH + f"{WATERSHED_NAME}_tp_tc.csv"
extract_subbasin_properties(rvh_path=FILE_PATH_RVH, output_csv=FILE_PATH_TP_TC_CSV)